# TP4 — Régression : estimer le prix des logements en Californie (Énoncé)

Cours associé : [TP4_Cours.md](TP4_Cours.md) · Pense-bête : [Pense_bete_Pandas.md](../Pense_bete_Pandas.md)

## Contexte

Une agence immobilière souhaite un outil d'estimation rapide du prix des logements. Vous disposez des données du recensement californien de 1990, agrégées par **quartier** (*block group*, quelques centaines à quelques milliers d'habitants). Pour 20 640 quartiers :

| Colonne | Signification |
|---|---|
| `MedInc` | revenu médian des ménages (en dizaines de milliers de $) |
| `HouseAge` | âge médian des logements (années) |
| `AveRooms`, `AveBedrms` | nombre moyen de pièces et de chambres par logement |
| `Population` | nombre d'habitants du quartier |
| `AveOccup` | nombre moyen d'occupants par logement |
| `Latitude`, `Longitude` | position du quartier |
| `MedHouseVal` | **cible** : valeur médiane des logements (en centaines de milliers de $) |

## Déroulé

| Partie | Contenu |
|---|---|
| 1 | Exploration des données |
| 2 | Baseline et régression linéaire simple |
| 3 | Régression linéaire multiple et analyse des résidus |
| 4 | Variables polynomiales et régularisation |
| 5 | Modèles à base d'arbres |
| 6 | Comparaison finale et analyse des erreurs |
| 7 | *Facultatif* : approfondissements |

Les cellules marquées `# TODO` sont à compléter. Les cellules ✍️ attendent une réponse rédigée.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from xgboost import XGBRegressor

plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42

## Partie 1 — Exploration des données

**1.1** Chargez `data/california_housing.csv` dans `df`. Affichez les dimensions, les valeurs manquantes et les statistiques descriptives. Repérez-vous des valeurs surprenantes ?

In [ ]:
# TODO : chargement et statistiques descriptives


**1.2** Tracez l'histogramme de la cible `MedHouseVal` (100 classes). Combien de quartiers ont exactement la valeur maximale ? Qu'en déduisez-vous ?

In [ ]:
# TODO : distribution de la cible


**1.3** Représentez les quartiers sur une carte : nuage de points `Longitude` / `Latitude`, colorés par `MedHouseVal` (`cmap="viridis"`, petits points semi-transparents). Où sont les logements les plus chers ?

In [ ]:
# TODO : carte des prix


**1.4** Calculez les corrélations de chaque variable avec la cible, triées. Tracez le nuage `MedInc` / `MedHouseVal`.

In [ ]:
# TODO : corrélations et nuage revenu / prix


✍️ **Votre réponse :**

…

## Partie 2 — Baseline et régression linéaire simple

**2.1** Séparez `X` (toutes les colonnes sauf la cible) et `y`, puis découpez en 80 % / 20 % avec `random_state=RANDOM_STATE`.

In [ ]:
# TODO : découpage


La fonction suivante calcule les métriques sur le jeu de test et mémorise les résultats. Elle convertit MAE et RMSE en **milliers de dollars** pour qu'ils soient parlants.

In [ ]:
resultats = {}

def evaluer(nom, modele, X_eval=None, y_eval=None):
    X_eval = X_test if X_eval is None else X_eval
    y_eval = y_test if y_eval is None else y_eval
    y_pred = modele.predict(X_eval)
    resultats[nom] = {
        "MAE (k$)": mean_absolute_error(y_eval, y_pred) * 100,
        "RMSE (k$)": root_mean_squared_error(y_eval, y_pred) * 100,
        "R²": r2_score(y_eval, y_pred),
    }
    return pd.DataFrame(resultats).T.round(3)

**2.2** Évaluez un `DummyRegressor()` (prédit toujours la moyenne), puis une `LinearRegression` n'utilisant **que** `MedInc`. Tracez la droite obtenue par-dessus le nuage de points du jeu de test. Interprétez la pente.

In [ ]:
# TODO : baseline et régression simple


✍️ **Votre réponse :**

…

## Partie 3 — Régression linéaire multiple et analyse des résidus

**3.1** Créez un pipeline `lin` (`StandardScaler` + `LinearRegression`) utilisant toutes les variables, entraînez-le et évaluez-le.

In [ ]:
# TODO : régression linéaire multiple


**3.2** Tracez côte à côte, pour le jeu de test : (a) le prix prédit en fonction du prix réel, avec la diagonale `y = x` ; (b) les résidus (`réel − prédit`) en fonction du prix prédit. Décrivez les défauts du modèle.

In [ ]:
# TODO : graphiques de diagnostic


✍️ **Votre réponse :**

…

## Partie 4 — Variables polynomiales et régularisation

**4.1** Ajoutez des variables polynomiales de degré 2 (`PolynomialFeatures(degree=2, include_bias=False)`) avant la standardisation. Combien de variables obtient-on ? Évaluez sur le test. Recommencez avec le degré 3. Que se passe-t-il ?

In [ ]:
# TODO : régression polynomiale


✍️ **Votre réponse :**

…

**4.2** Régularisez le modèle de degré 3 avec une régression **Ridge** dont le paramètre `α` est choisi par validation croisée : `RidgeCV(alphas=np.logspace(-2, 4, 25))`. Quel `α` est retenu ? Évaluez.

In [ ]:
# TODO : Ridge


✍️ **Votre réponse :**

…

## Partie 5 — Modèles à base d'arbres

**5.1** Entraînez une forêt aléatoire (`RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1)`) et un `XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8)`. Aucune standardisation n'est nécessaire. Évaluez-les.

In [ ]:
# TODO : forêt aléatoire et XGBoost


**5.2** Tracez l'importance des variables de la forêt aléatoire et du modèle XGBoost côte à côte. Pourquoi la latitude et la longitude sont-elles bien mieux exploitées ici que par la régression linéaire ?

In [ ]:
# TODO : importances


✍️ **Votre réponse :**

…

**5.3** Vérifiez la limite d'extrapolation des arbres : quelle est la valeur maximale prédite par XGBoost sur le test ? Et par la régression linéaire ? Construisez ensuite un quartier fictif avec un revenu médian de 25 (250 000 $, au-delà du maximum observé), les autres variables égales à la médiane : que prédisent la forêt aléatoire et la régression linéaire ?

In [ ]:
# TODO : extrapolation


✍️ **Votre réponse :**

…

## Partie 6 — Comparaison finale et analyse des erreurs

**6.1** Affichez le tableau `resultats` trié par RMSE.

In [ ]:
# TODO : tableau récapitulatif


**6.2** Pour le meilleur modèle, calculez l'erreur absolue moyenne par **tranche de prix réel** (`pd.cut` de `y_test` en tranches de 0 à 1, 1 à 2, … 4 à 5 et « plafond » pour les valeurs ≥ 5). Où le modèle se trompe-t-il le plus ?

In [ ]:
# TODO : erreurs par tranche de prix


**6.3** Rédigez une courte conclusion pour l'agence immobilière : modèle recommandé, précision attendue, limites.

✍️ **Votre réponse :**

…

## Partie 7 — *Facultatif* : approfondissements

*À faire si vous avez terminé le reste. Les questions sont indépendantes les unes des autres.*

**7.1** Affichez les coefficients du modèle `lin` (variables standardisées) sous forme de diagramme en barres. Quelles variables comptent le plus ? Le signe de `AveRooms` vous surprend-il ? Comment l'expliquer ?

In [ ]:
# TODO : coefficients


✍️ **Votre réponse :**

…

**7.2** Évaluez `lin` par validation croisée à 5 plis (`KFold` avec `shuffle=True`) sur le jeu d'entraînement : RMSE moyen et écart-type. Le résultat est-il stable d'un pli à l'autre ?

In [ ]:
# TODO : validation croisée


✍️ **Votre réponse :**

…

**7.3** Régularisez le modèle polynomial de degré 3 avec un **Lasso** au lieu d'un Ridge (`LassoCV(cv=5, max_iter=5000, n_jobs=-1)`). Combien de coefficients sont exactement nuls ? Affichez les 10 variables polynomiales de plus fort coefficient (en valeur absolue).

*Le Lasso est plus long à entraîner (une trentaine de secondes).*

In [ ]:
# TODO : Lasso
import warnings
from sklearn.exceptions import ConvergenceWarning
# On masque les avertissements de convergence du Lasso pour les très petites valeurs de α
warnings.filterwarnings("ignore", category=ConvergenceWarning)

✍️ **Votre réponse :**

…

## Pour aller plus loin (bonus)

- Entraînez la régression linéaire sur `log(MedHouseVal)` grâce à `TransformedTargetRegressor(regressor=..., func=np.log, inverse_func=np.exp)` : les résidus sont-ils plus homogènes ?
- Ajoutez une variable « distance au centre de San Francisco » et « distance au centre de Los Angeles » (coordonnées approximatives : (37,77 ; −122,42) et (34,05 ; −118,24)). La régression linéaire en profite-t-elle ?
- Excluez les quartiers au plafond de l'entraînement : l'erreur sur les autres quartiers diminue-t-elle ?

In [ ]:
# Bonus : distances aux grandes villes pour la régression linéaire
